# Stage 3 — DPO Preference Alignment
Continues from Stage 2 SFT model. Trains on chosen/rejected pairs using DPO.

**Runtime → T4 GPU.**

**Prerequisite:** Run `scripts/build_preference_dataset.py` after Stage 2 finishes, before opening this notebook.

## 1. Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = "/content/drive/MyDrive/domain-ai-assistant-finetuning"
os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/reports", exist_ok=True)
print("Project dir:", PROJECT_DIR)


Mounted at /content/drive
Project dir: /content/drive/MyDrive/domain-ai-assistant-finetuning


## 2. Install dependencies

In [ ]:
%%capture

# Install Unsloth and its compatible dependencies
!pip install -U --no-cache-dir "unsloth[colab-new]"

# Install remaining packages
!pip install -U bitsandbytes accelerate datasets


## 3. Load preference dataset

In [ ]:
import json
from datasets import Dataset

data_path = f"{PROJECT_DIR}/data/preference_dataset.jsonl"
rows = []
with open(data_path, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

print(f"Loaded {len(rows)} preference pairs")
assert len(rows) >= 50, f"Only {len(rows)} pairs -- need >= 50"
dataset = Dataset.from_list(rows)
print(dataset[0])


## 4. Load Stage 2 SFT model
**Important:** We do NOT call `get_peft_model` here. Stage 2 already saved a LoRA adapter. Loading it directly is correct. Calling `get_peft_model` again would stack a second adapter on top, which trains the wrong weights and produces garbage output.

In [ ]:
from unsloth import FastLanguageModel
import torch

STAGE2_PATH = f"{PROJECT_DIR}/checkpoints/stage2_sft_final"
MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=STAGE2_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
print("Stage 2 SFT model loaded for DPO. No new LoRA added -- training existing adapter.")


## 5. Format for DPOTrainer
DPOTrainer expects columns: `prompt`, `chosen`, `rejected`.

In [ ]:
PROMPT_TEMPLATE = (
    "Below is an instruction that describes a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Response:\n"
)


def format_dpo(example):
    return {
        "prompt": PROMPT_TEMPLATE.format(instruction=example["prompt"]),
        "chosen": example["chosen"],
        "rejected": example["rejected"],
    }

dpo_dataset = dataset.map(format_dpo)
print(dpo_dataset[0])


## 6. DPO Training
Low LR (5e-6) and 2 epochs — DPO overfits fast on 50 pairs.

In [ ]:
from trl import DPOTrainer, DPOConfig

CHECKPOINT_DIR = f"{PROJECT_DIR}/checkpoints/stage3_dpo"

dpo_config = DPOConfig(
    output_dir=CHECKPOINT_DIR,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    num_train_epochs=2,
    learning_rate=5e-6,
    beta=0.1,

    logging_steps=5,

    # Disable intermediate checkpointing
    save_strategy="no",

    optim="adamw_8bit",

    seed=42,
    report_to="none",

    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=512,
)

dpo_trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,   # New API (preferred)
)

dpo_trainer.train()


## 7. Save final DPO model to Drive

In [ ]:
SAVE_PATH = f"{PROJECT_DIR}/checkpoints/stage3_dpo_final"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print("Stage 3 DPO model saved to:", SAVE_PATH)


## 8. Run 10 test questions — copy output into `reports/final_evaluation.md`

In [ ]:
FastLanguageModel.for_inference(model)

TEST_QUESTIONS = ['How can I tell if my blood sugar is dangerously low?', 'Can I skip my metformin if I feel fine today?', "What's the difference between type 1 and type 2 diabetes?", 'Is it safe to exercise right after taking insulin?', 'Why am I so thirsty all the time?', 'Can type 2 diabetes be cured if I lose enough weight?', 'What should I do if my blood sugar reads 250?', 'Is it normal for blood sugar to vary day to day?', 'Can stress affect my blood sugar?', "What's considered a dangerously low blood sugar level?"]

for q in TEST_QUESTIONS:
    prompt = PROMPT_TEMPLATE.format(instruction=q)
    inputs = tokenizer([prompt], retur


                       n_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)
    answer = tokenizer.decode(output[0], skip_special_tokens=True).split("### Response:")[-1].strip()
    print(f"Q: {q}\nA: {answer}\n{'-'*60}")


## 9. Zip and download final model

In [ ]:
import shutil
zip_out = f"{PROJECT_DIR}/stage3_dpo_final"
shutil.make_archive(zip_out, "zip", SAVE_PATH)
print(f"Download from Drive: {zip_out}.zip")


In [ ]:
%cd /content/drive/MyDrive/domain-ai-assistant-finetuning

!python validate_assignment.py

In [ ]:
from unsloth import FastLanguageModel

PROJECT_DIR = "/content/drive/MyDrive/domain-ai-assistant-finetuning"

MODEL_PATH = f"{PROJECT_DIR}/checkpoints/stage3_dpo_final"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=1024,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

In [ ]:
def ask(question):
    prompt = PROMPT_TEMPLATE.format(instruction=question)

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=150
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = answer.split("### Response:")[-1].strip()

    print(answer)
    return answer

In [ ]:
questions = [
    "What is diabetes?",
    "What causes Type 2 diabetes?",
    "Can I stop taking insulin?",
    "What should I do if my sugar is 350?",
    "What is hypoglycemia?",
    "Can stress raise blood sugar?",
    "Can diabetes damage kidneys?",
    "Can diabetes be cured?",
    "How often should I check blood sugar?",
    "What foods should I avoid?"
]

with open(f"{PROJECT_DIR}/reports/dpo_model_answers.md", "w") as f:
    f.write("# DPO Model Evaluation\n\n")

    for q in questions:
        ans = ask(q)

        f.write(f"## Question\n{q}\n\n")
        f.write(f"### Answer\n{ans}\n\n")
        f.write("---\n\n")

print("Saved successfully!")

In [ ]:
!python src/inference.py "What is diabetes?"

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!ls /content/drive/MyDrive

In [ ]:
!ls /content/drive/MyDrive/domain-ai-assistant-finetuning

In [ ]:
!ls /content/drive/MyDrive/domain-ai-assistant-finetuning/checkpoints

In [ ]:
!pip install -q unsloth

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memory allocated:", torch.cuda.memory_allocated()/1024**3, "GB")
else:
    print("NO GPU")

CUDA available: False
NO GPU
